# "가격을 맞혀봐요!" 캡스톤 프로젝트

이번 주 목표 - Amazon 데이터 스크랩을 기반으로 상품 설명에서 가격을 예측하는 모델 만들기

설명만으로 상품 가격을 예측하는 모델입니다.

# 진행 순서

1일차: 데이터 수집 및 정제  
2일차: 데이터 전처리  
3일차: 평가, 기준 모델, 전통적 ML  
4일차: 딥러닝과 LLM  
5일차: 프론티어 모델 파인튜닝  

## 2일차: 데이터 전처리

오늘은 상품 정보를 표준 형식으로 재작성합니다.  
LLM이 이 작업에 탁월합니다!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">데이터 전처리·재작성의 비즈니스 가치</h2>
            <span style="color:#181;">LLM 덕분에 불과 몇 년 전만 해도 불가능하다고 여겨지던 작업이 손쉬워졌습니다.
            이 접근법은 거의 모든 비즈니스 분야에 적용할 수 있으며,
            5주차에 활용했던 고급 기법과도 유사합니다.</span>
        </td>
    </tr>
</table>

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

ModuleNotFoundError: No module named 'groq'

# 다음 셀에서 데이터셋을 선택합니다

`LITE_MODE = True`: 무료·빠른 버전 (학습 데이터 20,000개)

`LITE_MODE = False`: 강력한 전체 버전 (학습 데이터 800,000개)

## 이번 실습 비용 안내

HuggingFace에서 데이터셋 불러오기만 할 경우: $0 (무료)

라이트 데이터셋 전처리 실행 시: $1 미만

전체 데이터셋 전처리 실행 시: $30

In [ ]:
LITE_MODE = True

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
items[2].id

In [ ]:
# 모든 항목에 고유 ID 부여

for index, item in enumerate(items):
    item.id = index

In [ ]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
print(items[0].full)

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
MODEL = "openai/gpt-oss-20b"


In [ ]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
items[0]

In [ ]:
make_jsonl(items[0])

In [ ]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
result = groq.batches.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## 이 로직을 Batch 클래스로 구현했습니다

- 항목을 1,000개씩 그룹으로 나눕니다
- 각 그룹에 대해 배치 처리를 시작합니다
- 완료 후 결과를 모니터링하고 수집할 수 있습니다

## 비용 안내

Groq 사용 시 - 라이트 데이터셋은 $1 미만, 전체 데이터셋은 $30 미만이었습니다

하지만 비용을 전혀 내지 않아도 됩니다! 다음 실습에서 제가 전처리한 결과를 불러올 수 있습니다

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
# 요약(summary)이 없는 항목의 인덱스 출력
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Hub에 업로드할 때 불필요한 필드 제거

for item in items:
    item.full = None
    item.id = None

## 최종 데이터셋을 Hub에 업로드합니다

라이트 모드이면 라이트 데이터셋만 업로드합니다

전체 모드이면 두 데이터셋 모두 업로드합니다 (나중에 라이트 버전을 쓸 경우를 위해)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## 완성된 데이터셋입니다!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
